In [1]:
import torch
import math

from flatbuffers.packer import float32
from keras.src.ops import dtype

In [3]:
x = torch.rand(3,4)
print(x)

tensor([[0.0741, 0.2453, 0.5523, 0.0406],
        [0.5757, 0.2184, 0.8167, 0.3651],
        [0.7546, 0.1662, 0.7518, 0.3238]])


In [4]:
#adding bias vector to the matrix
y = x + torch.tensor([1, 1, 2, 1])
print(y)

tensor([[1.0741, 1.2453, 2.5523, 1.0406],
        [1.5757, 1.2184, 2.8167, 1.3651],
        [1.7546, 1.1662, 2.7518, 1.3238]])


In [5]:
#per channel image scaling
z = y * 2
print(z)

tensor([[2.1483, 2.4905, 5.1047, 2.0813],
        [3.1514, 2.4368, 5.6335, 2.7301],
        [3.5091, 2.3325, 5.5036, 2.6477]])


In [7]:
#Practicing broadcasting on images using the NCHW image shape
images = torch.randn(2, 3, 2, 2) # 2 images, 3 channels (rgb), 2 width, 2 height
print(images)

tensor([[[[ 0.4527,  0.1291],
          [ 0.0679, -0.3836]],

         [[-0.7889,  0.0768],
          [-0.3852,  0.0715]],

         [[-0.8026,  0.7643],
          [-2.3358,  1.7666]]],


        [[[-1.6119,  1.7898],
          [-0.2566,  1.8046]],

         [[-1.1746, -1.3000],
          [-0.3010,  0.1278]],

         [[ 0.5655, -0.9894],
          [ 1.2934,  1.3158]]]])


In [10]:
scales = torch.tensor([.5, 1, 2]) # scale red by .5, green by 1, and blue by 2 w a shape of (3,)
#imgs_scaled = images * scales
#print(imgs_scaled)
# scales * images fails because pytorch compares right to left and pads 1s to the left so (2, 3, 2, 2) doesn't match with (1, 1, 1, 3) lands in the width slot instead of channel slot which 2 != 3

In [11]:
scales = scales.view(1,3,1,1)
print(scales)

tensor([[[[0.5000]],

         [[1.0000]],

         [[2.0000]]]])


In [12]:
imgs_scaled = images * scales
print(imgs_scaled)

tensor([[[[ 0.2263,  0.0646],
          [ 0.0340, -0.1918]],

         [[-0.7889,  0.0768],
          [-0.3852,  0.0715]],

         [[-1.6052,  1.5285],
          [-4.6715,  3.5331]]],


        [[[-0.8059,  0.8949],
          [-0.1283,  0.9023]],

         [[-1.1746, -1.3000],
          [-0.3010,  0.1278]],

         [[ 1.1310, -1.9788],
          [ 2.5869,  2.6315]]]])


In [17]:
tensor_1 = torch.randn(3, 4)
tensor_ones = torch.ones(3,4)
tensor_zeros = torch.zeros(3,4)
torch.manual_seed(3)
print(f"the random seed is \n {tensor_1}, \n the one tensor is \n {tensor_ones}, \n the zero tensor is \n {tensor_zeros}")

the random seed is 
 tensor([[ 0.8033,  0.1748,  0.0890, -0.6137],
        [ 0.0462, -1.3683,  0.3375,  1.0111],
        [-1.4352,  0.9774,  0.5220,  1.2379]]), 
 the one tensor is 
 tensor([[1., 1., 1., 1.],
        [1., 1., 1., 1.],
        [1., 1., 1., 1.]]), 
 the zero tensor is 
 tensor([[0., 0., 0., 0.],
        [0., 0., 0., 0.],
        [0., 0., 0., 0.]])


In [19]:
torch.manual_seed(3)
tensor_2 = torch.randn(3, 4)
tensor_3 = torch.randn(2, 3) #what happens at different dimension?
print(f"tensor 2 with the same dimensions is {tensor_2}")
print(f"tensor 3 with less dimensions is {tensor_3}") #so the manual_seed only transfers to same successive randn tensors w same dimensions.

tensor 2 with the same dimensions is tensor([[ 0.8033,  0.1748,  0.0890, -0.6137],
        [ 0.0462, -1.3683,  0.3375,  1.0111],
        [-1.4352,  0.9774,  0.5220,  1.2379]])
tensor 3 with less dimensions is tensor([[-0.8646,  0.2990,  0.4192],
        [-0.0799,  0.9264,  0.8157]])


In [23]:
#declaring the data type
a = torch.randn((2, 3), dtype=float)
print(a)

tensor([[-0.3922,  0.1519, -1.1837],
        [ 0.5344, -1.4510, -0.6294]], dtype=torch.float64)


In [27]:
#changing the dtype
b = a.to(int)
print(b)

tensor([[ 0,  0, -1],
        [ 0, -1,  0]])


In [38]:
#more broadcasting | Rules are that each dimension must either match or be 1 or not exist (treated as a 1)
one_tensor = torch.ones(3, 3, 3)
#print(one_tensor)

three_by_3_matrix = torch.tensor(data=[[1, 1, 1],[2, 2, 2],[3, 3, 3]])
two_by_3_matrix = torch.tensor(data=[[1, 1, 1],[2, 2, 2]])
three_by_1_matrix = torch.tensor(data=[[2], [2], [2]])

same_last_2_dims = one_tensor * three_by_3_matrix
#print(same_last_2_dims)

#The following returns a runtime error because the middle dimension is neither 3 nor 1 in (1, 2, 3).
# same_last_dims = one_tensor * two_by_3_matrix
# print(same_last_dims)

same_2nd_to_last_dims = one_tensor * three_by_1_matrix
print(same_2nd_to_last_dims)

tensor([[[2., 2., 2.],
         [2., 2., 2.],
         [2., 2., 2.]],

        [[2., 2., 2.],
         [2., 2., 2.],
         [2., 2., 2.]],

        [[2., 2., 2.],
         [2., 2., 2.],
         [2., 2., 2.]]])


In [39]:
#Check to see if there is an accelerator available
if torch.accelerator.is_available():
    gpu_rand = torch.rand(2, 2, device=torch.accelerator.current_accelerator())
    print(gpu_rand)
else:
    print("Sorry, CPU only.")

Sorry, CPU only.


In [41]:
#Let's say we have an image that we want to its shape is 3d since there is only 1 [n, n, n] where PyTorch is expecting a [n, n, n, n]
img = torch.rand(3, 200, 200) #1 img, 3 channels rgb, 200 h, 200 w
new_img_shape = img.unsqueeze(0) #add 1 to the 0th index
print(new_img_shape.shape)

torch.Size([1, 3, 200, 200])


In [42]:
new_img_shape2 = new_img_shape.squeeze(0)
print(new_img_shape2.shape)

torch.Size([3, 200, 200])
